# HAM10000 — Stage 6: Cross-model Comparison

**Goal:** Load `outputs/metrics/*.json` (written by Stages 3 and 4) and produce
a clean comparison table — no model loading, no GPU required.

**Prerequisites:** Run notebooks 03 → 04 on Colab first, download the output zips,
and commit `outputs/metrics/*.json` to the repo. Then run this notebook.

If any JSON is missing, that model's row shows `—` and a warning is printed.

## Cell 1 — Clone / update repo and set up paths

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Dev252001/HAM10000.git"
REPO_DIR = "/content/ham10000-classifier"
SRC_DIR  = os.path.join(REPO_DIR, "src")
OUT_DIR  = os.path.join(REPO_DIR, "outputs")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Repo updated.")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"src/ on path: {SRC_DIR}")

## Cell 2 — Install dependencies

In [ ]:
!pip install -q \
  "pandas>=2.2.2" \
  "matplotlib>=3.9.0"
print("Dependencies ready.")

## Cell 3 — Load metrics JSONs

Reads every `*.json` in `outputs/metrics/`. Rows with missing files show `—`.

In [ ]:
import json
import os
import glob
import pandas as pd

METRICS_DIR = os.path.join(OUT_DIR, "metrics")

json_files = sorted(glob.glob(os.path.join(METRICS_DIR, "*.json")))

if not json_files:
    print(f"⚠️  No JSON files found in {METRICS_DIR}")
    print("   Run notebooks 03 and 04 on Colab first, then commit outputs/metrics/*.json")
else:
    print(f"Found {len(json_files)} metric file(s):")
    for f in json_files:
        print(f"  {os.path.basename(f)}")

all_metrics = {}
for fpath in json_files:
    with open(fpath) as f:
        data = json.load(f)
    key = f"{data['model_name']}_{data['split_mode']}_seed{data['seed']}"
    all_metrics[key] = data

print(f"\nLoaded: {list(all_metrics.keys())}")

## Cell 4 — Primary comparison table (lesion split)

Primary results: all three models on the lesion-level split.

In [ ]:
DISPLAY_ORDER = [
    ("baseline_cnn",    "lesion", 42, "Baseline CNN (from scratch)"),
    ("resnet18",        "lesion", 42, "ResNet18 (fine-tuned)"),
    ("efficientnet_b0", "lesion", 42, "EfficientNet-B0 (fine-tuned)"),
]

def fmt(v, fmt_str):
    return "—" if v is None else format(v, fmt_str)

rows = []
for model_name, split_mode, seed, label in DISPLAY_ORDER:
    key  = f"{model_name}_{split_mode}_seed{seed}"
    data = all_metrics.get(key)
    if data is None:
        print(f"⚠️  Missing: {key}.json — run notebook 03 or 04 first")
    rows.append({
        "Model"           : label,
        "Split"           : split_mode,
        "Accuracy"        : fmt(data["accuracy"]     if data else None, ".2%"),
        "Macro F1"        : fmt(data["macro_f1"]     if data else None, ".4f"),
        "Weighted F1"     : fmt(data["weighted_f1"]  if data else None, ".4f"),
        "Recall — mel"    : fmt(data["malignant_recall"].get("mel")   if data else None, ".4f"),
        "Recall — bcc"    : fmt(data["malignant_recall"].get("bcc")   if data else None, ".4f"),
        "Recall — akiec"  : fmt(data["malignant_recall"].get("akiec") if data else None, ".4f"),
        "Params"          : fmt(data["n_params"]      if data else None, ",d"),
        "Train time (s)"  : fmt(data["train_time_s"] if data else None, ".0f"),
    })

df_primary = pd.DataFrame(rows)
print("=" * 80)
print("PRIMARY RESULTS — Lesion-level split (split_mode='lesion', seed=42)")
print("=" * 80)
print(df_primary.to_string(index=False))
print()
print("Note: '—' means the JSON file is missing. Run the corresponding notebook first.")

## Cell 5 — Leakage comparison (image split vs lesion split)

If both `*_lesion_seed42.json` and `*_image_seed42.json` exist for a model,
shows the accuracy difference (image − lesion) to quantify how much the
image-level split inflated the score. Skipped gracefully if not available.

In [ ]:
leakage_rows = []
for model_name, _, _, label in DISPLAY_ORDER:
    key_lesion = f"{model_name}_lesion_seed42"
    key_image  = f"{model_name}_image_seed42"
    d_l = all_metrics.get(key_lesion)
    d_i = all_metrics.get(key_image)
    if d_l is None or d_i is None:
        continue
    leakage_rows.append({
        "Model"              : label,
        "Acc (lesion)"       : f"{d_l['accuracy']:.2%}",
        "Acc (image)"        : f"{d_i['accuracy']:.2%}",
        "Acc diff (img−les)" : f"{(d_i['accuracy'] - d_l['accuracy'])*100:+.1f} pp",
        "F1 (lesion)"        : f"{d_l['macro_f1']:.4f}",
        "F1 (image)"         : f"{d_i['macro_f1']:.4f}",
        "F1 diff (img−les)"  : f"{d_i['macro_f1'] - d_l['macro_f1']:+.4f}",
    })

if leakage_rows:
    df_leak = pd.DataFrame(leakage_rows)
    print("=" * 80)
    print("LEAKAGE COMPARISON — Image split vs Lesion split")
    print("Positive diff = image split inflated the score")
    print("=" * 80)
    print(df_leak.to_string(index=False))
else:
    print("Leakage comparison not available — need both *_lesion_seed42.json and")
    print("*_image_seed42.json for at least one model.")
    print("Run notebooks 03/04 with split_mode='image' to generate image-split results.")

## Cell 6 — Save model_comparison.csv

In [ ]:
csv_path = os.path.join(OUT_DIR, "model_comparison.csv")
df_primary.to_csv(csv_path, index=False)
print(f"Saved → {csv_path}")

## Cell 7 — Download outputs

In [ ]:
import shutil
from google.colab import files as colab_files

# Zip metrics dir + the CSV
zip_path = "/content/outputs_comparison.zip"
shutil.make_archive("/content/outputs_comparison", "zip", OUT_DIR, "metrics")

# Also include model_comparison.csv at the root
import zipfile
with zipfile.ZipFile(zip_path, 'a') as zf:
    zf.write(csv_path, arcname="model_comparison.csv")

print(f"Created: {zip_path}")
colab_files.download(zip_path)

print("\nAfter downloading:")
print("  1. Unzip into ham10000-classifier/outputs/")
print("  2. git add outputs/metrics/ outputs/model_comparison.csv")
print("  3. git commit -m 'results: add Stage 6 comparison CSV'")